# Module 4 — Exploratory Data Analysis
Reuses `03_Cleaned_Data` + `src/eda_utils.py`.

In [122]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd() / "src"))
from eda_utils import (
    REPORT_DIR, CHART_DIR, SPENDING_COLS, PURCHASE_COLS, CAMPAIGN_COLS,
    load_cleaned_data, save_fig, save_excel,
    top_correlations, add_engineered_columns, add_segments,
)

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")


In [123]:
df = load_cleaned_data()
df = add_engineered_columns(df)
df.head()


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Enrollment_Year,Enrollment_Month,Age,Total_Spending,Total_Purchases
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,1,2012,9,57,1617,25
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,0,2014,3,60,27,6
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,0,2013,8,49,776,21
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,0,2014,2,30,53,8
4,5324,1981,Phd,Married,58293.0,1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,0,2014,1,33,422,19


## 1. Dataset Overview

In [124]:
shape = df.shape
dtypes = df.dtypes.astype(str)
missing = df.isnull().sum()
duplicates = df.duplicated().sum()

print(shape)
df.info()


(2237, 32)
<class 'pandas.DataFrame'>
RangeIndex: 2237 entries, 0 to 2236
Data columns (total 32 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID                   2237 non-null   int64         
 1   Year_Birth           2237 non-null   int64         
 2   Education            2237 non-null   str           
 3   Marital_Status       2237 non-null   str           
 4   Income               2237 non-null   float64       
 5   Kidhome              2237 non-null   int64         
 6   Teenhome             2237 non-null   int64         
 7   Dt_Customer          2237 non-null   datetime64[us]
 8   Recency              2237 non-null   int64         
 9   MntWines             2237 non-null   int64         
 10  MntFruits            2237 non-null   int64         
 11  MntMeatProducts      2237 non-null   int64         
 12  MntFishProducts      2237 non-null   int64         
 13  MntSweetProducts     2237 non-nul

In [125]:
df.describe()

,ID,Year_Birth,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Enrollment_Year,Enrollment_Month,Age,Total_Spending,Total_Purchases
count,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000,2237.000000
mean,5590.726419,1968.901654,51854.814037,0.444345,0.506482,2013-07-10 05:01:54.260169,49.104604,303.995530,26.270451,166.916853,37.523022,27.068842,43.968708,2.326777,4.087170,2.662494,5.794367,5.319177,0.072865,0.074654,0.072418,0.064372,0.013411,0.008941,0.149307,2013.027716,6.465802,45.098346,605.743406,14.870809
min,0.000000,1940.000000,1730.000000,0.000000,0.000000,2012-07-30 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2012.000000,1.000000,18.000000,5.000000,0.000000
25%,2829.000000,1959.000000,35523.000000,0.000000,0.000000,2013-01-16 00:00:00,24.000000,24.000000,1.000000,16.000000,3.000000,1.000000,9.000000,1.000000,2.000000,0.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2013.000000,3.000000,37.000000,69.000000,8.000000
50%,5455.000000,1970.000000,51381.500000,0.000000,0.000000,2013-07-08 00:00:00,49.000000,174.000000,8.000000,67.000000,12.000000,8.000000,24.000000,2.000000,4.000000,2.000000,5.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2013.000000,6.000000,44.000000,396.000000,15.000000
75%,8427.000000,1977.000000,68281.000000,1.000000,1.000000,2013-12-30 00:00:00,74.000000,504.000000,33.000000,232.000000,50.000000,33.000000,56.000000,3.000000,6.000000,4.000000,8.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2013.000000,10.000000,55.000000,1045.000000,21.000000
max,11191.000000,1996.000000,117418.000000,2.000000,2.000000,2014-06-29 00:00:00,99.000000,1493.000000,199.000000,1725.000000,259.000000,263.000000,362.000000,15.000000,27.000000,28.000000,13.000000,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2014.000000,12.000000,74.000000,2525.000000,44.000000
std,3245.118591,11.701917,20936.241476,0.538467,0.544593,NaN,28.956073,336.574382,39.715972,225.661158,54.639909,41.293949,52.054318,1.932923,2.779461,2.923456,3.250940,2.426386,0.259974,0.262890,0.259237,0.245469,0.115052,0.094152,0.356471,0.684704,3.488073,11.701917,601.840466,7.676593


In [126]:
dataset_summary = pd.DataFrame({
    "Column": df.columns,
    "Dtype": dtypes.reindex(df.columns).values,
    "Missing": missing.reindex(df.columns).values,
})
dataset_summary.loc["__meta__"] = ["Rows", shape[0], "Duplicates: " + str(duplicates)]
save_excel(dataset_summary, "dataset_summary")
dataset_summary


,Column,Dtype,Missing
0,ID,int64,0
1,Year_Birth,int64,0
2,Education,str,0
3,Marital_Status,str,0
4,Income,float64,0
5,Kidhome,int64,0
6,Teenhome,int64,0
7,Dt_Customer,datetime64[us],0
8,Recency,int64,0
9,MntWines,int64,0


## 2. Demographic Analysis

In [127]:
fig, ax = plt.subplots()
ax.hist(df["Age"], bins=20, color="steelblue")
ax.set_title("Age Distribution")
save_fig(fig, "age_histogram")

fig, ax = plt.subplots()
ax.boxplot(df["Income"])
ax.set_title("Income Boxplot")
save_fig(fig, "income_boxplot")

fig, ax = plt.subplots()
sns.countplot(data=df, x="Education", ax=ax, order=df["Education"].value_counts().index)
ax.set_title("Education Countplot")
save_fig(fig, "education_countplot")

fig, ax = plt.subplots()
df["Marital_Status"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax)
ax.set_ylabel("")
ax.set_title("Marital Status Share")
save_fig(fig, "marital_status_pie")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x="Kidhome", ax=axes[0])
sns.countplot(data=df, x="Teenhome", ax=axes[1])
axes[0].set_title("Kidhome")
axes[1].set_title("Teenhome")
save_fig(fig, "kidhome_teenhome_countplot")


In [128]:
demographic_summary = pd.DataFrame({
    "Metric": ["Age Mean", "Age Median", "Income Mean", "Income Median",
               "Kidhome Mean", "Teenhome Mean"],
    "Value": [df["Age"].mean(), df["Age"].median(), df["Income"].mean(),
              df["Income"].median(), df["Kidhome"].mean(), df["Teenhome"].mean()],
})
save_excel(demographic_summary, "demographic_summary")
demographic_summary


,Metric,Value
0,Age Mean,45.098346
1,Age Median,44.000000
2,Income Mean,51854.814037
3,Income Median,51381.500000
4,Kidhome Mean,0.444345
5,Teenhome Mean,0.506482


## 3. Spending Analysis
`Total_Spending` = sum of all `Mnt*` columns.

In [129]:
category_spending = df[SPENDING_COLS].sum().sort_values(ascending=False)
avg_spending = df["Total_Spending"].mean()
top_spender = df.loc[df["Total_Spending"].idxmax(), ["ID", "Total_Spending"]]

print("Average Total_Spending:", round(avg_spending, 2))
print("Top spender:\n", top_spender)
category_spending


Average Total_Spending: 605.74
Top spender:
 ID                5735
Total_Spending    2525
Name: 1176, dtype: int64


MntWines            680038
MntMeatProducts     373393
MntGoldProds         98358
MntFishProducts      83939
MntSweetProducts     60553
MntFruits            58767
dtype: int64

In [130]:
fig, ax = plt.subplots()
category_spending.plot.bar(ax=ax)
ax.set_title("Total Spending by Category")
save_fig(fig, "spending_by_category_bar")

fig, ax = plt.subplots()
ax.hist(df["Total_Spending"], bins=30, color="darkorange")
ax.set_title("Total Spending Distribution")
save_fig(fig, "total_spending_histogram")

fig, ax = plt.subplots()
ax.boxplot(df["Total_Spending"])
ax.set_title("Total Spending Boxplot")
save_fig(fig, "total_spending_boxplot")

edu_spending = df.groupby("Education")[SPENDING_COLS].sum()
fig, ax = plt.subplots()
edu_spending.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Spending by Category per Education (Stacked)")
save_fig(fig, "spending_stacked_bar")


In [131]:
spending_summary = category_spending.reset_index()
spending_summary.columns = ["Category", "Total_Spending"]
spending_summary.loc[len(spending_summary)] = ["Average_Total_Spending", avg_spending]
save_excel(spending_summary, "spending_summary")
spending_summary


,Category,Total_Spending
0,MntWines,680038.000000
1,MntMeatProducts,373393.000000
2,MntGoldProds,98358.000000
3,MntFishProducts,83939.000000
4,MntSweetProducts,60553.000000
5,MntFruits,58767.000000
6,Average_Total_Spending,605.743406


## 4. Purchasing Behaviour
`Total_Purchases` = sum of Web/Store/Catalog/Deals purchases.

In [132]:
channel_totals = df[PURCHASE_COLS].sum().sort_values(ascending=False)
channel_totals


NumStorePurchases      12962
NumWebPurchases         9143
NumCatalogPurchases     5956
NumDealsPurchases       5205
dtype: int64

In [133]:
fig, ax = plt.subplots()
channel_totals.plot.bar(ax=ax)
ax.set_title("Purchases by Channel")
save_fig(fig, "purchases_by_channel_bar")

fig, ax = plt.subplots()
ax.hist(df["Total_Purchases"], bins=20, color="seagreen")
ax.set_title("Total Purchases Distribution")
save_fig(fig, "total_purchases_histogram")

fig, ax = plt.subplots()
sns.countplot(data=df, x="NumDealsPurchases", ax=ax)
ax.set_title("Deal Purchases Countplot")
save_fig(fig, "deals_countplot")


In [134]:
purchase_summary = channel_totals.reset_index()
purchase_summary.columns = ["Channel", "Total"]
purchase_summary.loc[len(purchase_summary)] = ["Avg_Total_Purchases", df["Total_Purchases"].mean()]
save_excel(purchase_summary, "purchase_summary")
purchase_summary


,Channel,Total
0,NumStorePurchases,12962.000000
1,NumWebPurchases,9143.000000
2,NumCatalogPurchases,5956.000000
3,NumDealsPurchases,5205.000000
4,Avg_Total_Purchases,14.870809


## 5. Website Engagement

In [135]:
engagement_median = df["NumWebVisitsMonth"].median()
high_engagement = (df["NumWebVisitsMonth"] > engagement_median).sum()
low_engagement = (df["NumWebVisitsMonth"] <= engagement_median).sum()

fig, ax = plt.subplots()
ax.hist(df["NumWebVisitsMonth"], bins=15, color="slateblue")
ax.set_title("Web Visits per Month")
save_fig(fig, "webvisits_histogram")

fig, ax = plt.subplots()
ax.scatter(df["NumWebVisitsMonth"], df["Total_Spending"], alpha=0.4)
ax.set_xlabel("NumWebVisitsMonth")
ax.set_ylabel("Total_Spending")
ax.set_title("Web Visits vs Spending")
save_fig(fig, "webvisits_scatter")

fig, ax = plt.subplots()
sns.kdeplot(df["NumWebVisitsMonth"], ax=ax, fill=True)
ax.set_title("Web Visits KDE")
save_fig(fig, "webvisits_kde")


In [136]:
website_summary = pd.DataFrame({
    "Metric": ["Median Visits", "High Engagement Count", "Low Engagement Count"],
    "Value": [engagement_median, high_engagement, low_engagement],
})
save_excel(website_summary, "website_summary")
website_summary


,Metric,Value
0,Median Visits,6.0
1,High Engagement Count,830.0
2,Low Engagement Count,1407.0


## 6. Recency Analysis

In [137]:
recency_threshold = 30
active_customers = (df["Recency"] <= recency_threshold).sum()
inactive_customers = (df["Recency"] > recency_threshold).sum()

fig, ax = plt.subplots()
ax.hist(df["Recency"], bins=20, color="teal")
ax.set_title("Recency Distribution")
save_fig(fig, "recency_histogram")

print(f"Active (<= {recency_threshold} days): {active_customers}")
print(f"Inactive (> {recency_threshold} days): {inactive_customers}")
print("Recommendation: target inactive customers with re-engagement offers/discounts.")


Active (<= 30 days): 723
Inactive (> 30 days): 1514
Recommendation: target inactive customers with re-engagement offers/discounts.


## 7. Campaign Analysis

In [138]:
campaign_all_cols = CAMPAIGN_COLS + ["Response"]
acceptance_rates = df[campaign_all_cols].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots()
acceptance_rates.plot.bar(ax=ax)
ax.set_ylabel("Acceptance Rate (%)")
ax.set_title("Campaign Acceptance Rates")
save_fig(fig, "campaign_acceptance_bar")

fig, ax = plt.subplots()
sns.countplot(data=df, x="Response", ax=ax)
ax.set_title("Response Countplot")
save_fig(fig, "response_countplot")

fig, ax = plt.subplots()
df["Response"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax)
ax.set_ylabel("")
ax.set_title("Response Share")
save_fig(fig, "response_pie")


In [139]:
campaign_summary = acceptance_rates.reset_index()
campaign_summary.columns = ["Campaign", "Acceptance_Rate_%"]
save_excel(campaign_summary, "campaign_summary")
campaign_summary


,Campaign,Acceptance_Rate_%
0,Response,14.930711
1,AcceptedCmp4,7.465355
2,AcceptedCmp3,7.286544
3,AcceptedCmp5,7.241842
4,AcceptedCmp1,6.437193
5,AcceptedCmp2,1.341082


## 8. Complaint Analysis

In [140]:
complaint_pct = df["Complain"].mean() * 100
complaint_vs_spending = df.groupby("Complain")["Total_Spending"].mean()
complaint_vs_income = df.groupby("Complain")["Income"].mean()

fig, ax = plt.subplots()
sns.countplot(data=df, x="Complain", ax=ax)
ax.set_title("Complaint Countplot")
save_fig(fig, "complaint_countplot")

heat_data = df.groupby("Complain")[["Total_Spending", "Income"]].mean()
fig, ax = plt.subplots()
sns.heatmap(heat_data.T, annot=True, fmt=".0f", cmap="coolwarm", ax=ax)
ax.set_title("Complaint vs Spending/Income")
save_fig(fig, "complaint_heatmap")


In [141]:
complaint_summary = pd.DataFrame({
    "Complain": complaint_vs_spending.index,
    "Avg_Spending": complaint_vs_spending.values,
    "Avg_Income": complaint_vs_income.values,
})
complaint_summary.loc[len(complaint_summary)] = ["Complaint_%", complaint_pct, np.nan]
save_excel(complaint_summary, "complaint_summary")
complaint_summary


,Complain,Avg_Spending,Avg_Income
0,0,607.671628,51910.586829
1,1,392.000000,45672.400000
2,Complaint_%,0.894055,NaN


## 9. Correlation Analysis

In [142]:
numeric_df = df.select_dtypes(include=np.number).drop(columns=["ID"])
corr = numeric_df.corr(method="pearson")

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix (Pearson)")
save_fig(fig, "correlation_heatmap")

top_pos, top_neg = top_correlations(corr, n=5)
print(top_pos)
print(top_neg)


MntWines             Total_Spending     0.891734
MntMeatProducts      Total_Spending     0.842655
NumStorePurchases    Total_Purchases    0.820089
Income               Total_Spending     0.804182
NumCatalogPurchases  Total_Spending     0.778375
dtype: float64
Year_Birth       Age                 -1.000000
Income           NumWebVisitsMonth   -0.648292
Enrollment_Year  Enrollment_Month    -0.620737
Kidhome          Total_Spending      -0.556902
MntMeatProducts  NumWebVisitsMonth   -0.539203
dtype: float64


In [143]:
correlation_summary = pd.concat([
    top_pos.rename("Correlation").reset_index().assign(Type="Top Positive"),
    top_neg.rename("Correlation").reset_index().assign(Type="Top Negative"),
], ignore_index=True)
correlation_summary.columns = ["Var1", "Var2", "Correlation", "Type"]
save_excel(correlation_summary, "correlation_summary")
correlation_summary


,Var1,Var2,Correlation,Type
0,MntWines,Total_Spending,0.891734,Top Positive
1,MntMeatProducts,Total_Spending,0.842655,Top Positive
2,NumStorePurchases,Total_Purchases,0.820089,Top Positive
3,Income,Total_Spending,0.804182,Top Positive
4,NumCatalogPurchases,Total_Spending,0.778375,Top Positive
5,Year_Birth,Age,-1.000000,Top Negative
6,Income,NumWebVisitsMonth,-0.648292,Top Negative
7,Enrollment_Year,Enrollment_Month,-0.620737,Top Negative
8,Kidhome,Total_Spending,-0.556902,Top Negative
9,MntMeatProducts,NumWebVisitsMonth,-0.539203,Top Negative


## 10. Customer Segmentation

In [144]:
segments = add_segments(df)
segment_counts = segments.sum().rename("Count").reset_index()
segment_counts.columns = ["Segment", "Count"]
segment_counts["Percent"] = (segment_counts["Count"] / len(df) * 100).round(2)
save_excel(segment_counts, "customer_segments")
segment_counts


,Segment,Count,Percent
0,High_Value,559,24.99
1,Low_Value,558,24.94
2,Frequent_Buyer,512,22.89
3,Campaign_Responder,608,27.18
4,Inactive,550,24.59
5,Discount_Seeker,430,19.22


## 11. Business Insights

In [145]:
best_channel = channel_totals.idxmax()
best_campaign = acceptance_rates.idxmax()
income_spending_corr = df["Income"].corr(df["Total_Spending"])
children_spending_corr = (df["Kidhome"] + df["Teenhome"]).corr(df["Total_Spending"])
high_value_profile = df.loc[segments["High_Value"], ["Age", "Income", "Total_Spending"]].mean()

insights = pd.DataFrame({
    "Insight": [
        "Highest spending customer (ID)",
        "Highest revenue category",
        "Income vs Spending correlation",
        "Most popular purchase channel",
        "Best performing campaign",
        "Most responsive segment",
        "Inactive customers (count)",
        "High engagement customers (count)",
        "Children vs Spending correlation",
        "High-Value avg Age",
        "High-Value avg Income",
        "High-Value avg Spending",
        "Strongest positive correlation",
        "Strongest negative correlation",
        "Marketing recommendation",
    ],
    "Value": [
        int(top_spender["ID"]),
        category_spending.idxmax(),
        round(income_spending_corr, 3),
        best_channel,
        best_campaign,
        "Campaign_Responder segment (see customer_segments.xlsx)",
        int(inactive_customers),
        int(high_engagement),
        round(children_spending_corr, 3),
        round(high_value_profile["Age"], 1),
        round(high_value_profile["Income"], 2),
        round(high_value_profile["Total_Spending"], 2),
        f"{top_pos.index[0]} ({top_pos.iloc[0]:.2f})",
        f"{top_neg.index[0]} ({top_neg.iloc[0]:.2f})",
        "Prioritize wine/meat buyers, re-target inactive customers, "
        "and expand the best-performing campaign to high-value segments.",
    ],
})
save_excel(insights, "business_insights")
insights


,Insight,Value
0,Highest spending customer (ID),5735
1,Highest revenue category,MntWines
2,Income vs Spending correlation,0.804
3,Most popular purchase channel,NumStorePurchases
4,Best performing campaign,Response
5,Most responsive segment,Campaign_Responder segment (see customer_segme...
6,Inactive customers (count),1514
7,High engagement customers (count),830
8,Children vs Spending correlation,-0.499
9,High-Value avg Age,46.0
